# Computer Vision — Part 5
## Optical Flow: Lucas-Kanade vs FlowNet
### Student Notebook

---

**What you will do:**
1. Detect features and run Lucas-Kanade optical flow on synthetic and real frames
2. Load a pretrained FlowNet and run dense optical flow inference
3. Tune LK parameters and observe the effect on tracking quality
4. Run a forward-backward consistency check on FlowNet
5. Build a motion segmentation mask from dense flow
6. Compare both methods quantitatively (EPE) and explain the results

> All deep learning uses **pretrained weights** — no training from scratch.

---

## Setup

In [ ]:
!pip install torch torchvision opencv-python matplotlib numpy ptlflow --quiet

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import ptlflow
from pathlib import Path
import urllib.request

print(f"OpenCV : {cv2.__version__}")
print(f"PyTorch: {torch.__version__}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device : {DEVICE}")

## Utilities (provided)

In [ ]:
def flow_to_color(flow: np.ndarray) -> np.ndarray:
    """(H,W,2) flow → BGR color image. Hue=direction, Value=magnitude."""
    hsv = np.zeros((*flow.shape[:2], 3), dtype=np.uint8)
    hsv[..., 1] = 255
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

def show_side_by_side(*imgs_titles, figsize=(18, 5)):
    fig, axes = plt.subplots(1, len(imgs_titles), figsize=figsize)
    if len(imgs_titles) == 1:
        axes = [axes]
    for ax, (img, title) in zip(axes, imgs_titles):
        disp = img if len(img.shape) == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(disp, cmap='gray' if len(img.shape) == 2 else None)
        ax.set_title(title, fontsize=12)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def epe_map(flow, u_gt, v_gt):
    return np.sqrt((flow[..., 0] - u_gt)**2 + (flow[..., 1] - v_gt)**2)

print("Utilities ready.")

## Section 1 — Test Frames

We use two kinds of input throughout:
- **Synthetic**: white ball on black background with known ground-truth motion `(GT_U, GT_V)`
- **Real**: two consecutive frames from a surveillance video

In [ ]:
GT_U, GT_V = 20, 8

def make_ball(cx, cy, h=300, w=400):
    img = np.full((h, w), 25, dtype=np.uint8)
    cv2.circle(img, (cx, cy), 35, 230, -1)
    return img

synth1 = make_ball(180, 140)
synth2 = make_ball(180 + GT_U, 140 + GT_V)

show_side_by_side(
    (synth1, 'Synth Frame 1'),
    (synth2, f'Synth Frame 2  (GT: u={GT_U}, v={GT_V})')
)

VIDEO_URL  = "https://github.com/opencv/opencv/raw/master/samples/data/vtest.avi"
VIDEO_PATH = "vtest.avi"
if not Path(VIDEO_PATH).exists():
    urllib.request.urlretrieve(VIDEO_URL, VIDEO_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)
_, real1 = cap.read()
_, real2 = cap.read()
_, real3 = cap.read()   # we'll use a 3rd frame later
cap.release()

real1_gray = cv2.cvtColor(real1, cv2.COLOR_BGR2GRAY)
real2_gray = cv2.cvtColor(real2, cv2.COLOR_BGR2GRAY)
real3_gray = cv2.cvtColor(real3, cv2.COLOR_BGR2GRAY)

show_side_by_side((real1, 'Real Frame 1'), (real2, 'Real Frame 2'))

## Section 2 — Lucas-Kanade Optical Flow

LK tracks **corners** detected by Shi-Tomasi. It solves $x = (A^TA)^{-1}A^Tb$ over a local pixel window.

### 2.1 — Synthetic frames

In [ ]:
LK_PARAMS = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
)

# TODO: detect corners in synth1 using cv2.goodFeaturesToTrack
# Parameters: maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7
corners = # YOUR CODE HERE

# TODO: track corners from synth1 to synth2 using cv2.calcOpticalFlowPyrLK
# Use corners and LK_PARAMS defined above
new_corners, status, _ = # YOUR CODE HERE

good_old = corners[status == 1]
good_new = new_corners[status == 1]

disp = good_new - good_old
mean_u, mean_v = disp[:, 0].mean(), disp[:, 1].mean()
print(f"LK estimated:  u={mean_u:.2f}, v={mean_v:.2f}")
print(f"Ground truth:  u={GT_U},       v={GT_V}")
print(f"Error:         Δu={abs(mean_u-GT_U):.3f}, Δv={abs(mean_v-GT_V):.3f}")

In [ ]:
# TODO: visualize the tracked vectors on synth1
# Draw a green arrowed line from each old corner to its new position
# Draw a small red circle at each old corner
# Show with show_side_by_side

lk_vis_synth = cv2.cvtColor(synth1, cv2.COLOR_GRAY2BGR)

# YOUR CODE HERE

show_side_by_side(
    (synth1, 'Frame 1'),
    (synth2, 'Frame 2'),
    (lk_vis_synth, f'LK — {len(good_old)} tracked corners')
)

### 2.2 — Real video frames

In [ ]:
# TODO: repeat the full LK pipeline on real video frames
# 1. Detect corners in real1_gray (same parameters as above)
# 2. Track to real2_gray
# 3. Draw arrows on a copy of real1 and display

real_corners = # YOUR CODE HERE
real_new_corners, real_status, _ = # YOUR CODE HERE

r_good_old = real_corners[real_status == 1]
r_good_new = real_new_corners[real_status == 1]

lk_vis_real = real1.copy()

# YOUR CODE HERE (draw arrows and circles)

show_side_by_side((lk_vis_real, f'LK — Real Video  ({len(r_good_old)} tracked pts)'))

### 2.3 — Parameter tuning

The `winSize` and `maxLevel` parameters directly affect what LK can and cannot track.
Run the experiments below and observe.

In [ ]:
# TODO: Run LK with three different winSize values on the synthetic frames:
#   (5,5), (15,15), (31,31)
# For each: compute mean EPE at tracked corners, count tracked points.
# Display all three arrow visualizations side by side.
# Then answer in a comment: what is the trade-off of a larger window?

for win in [(5, 5), (15, 15), (31, 31)]:
    params = dict(winSize=win, maxLevel=2,
                  criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))
    # YOUR CODE HERE
    pass

In [ ]:
# TODO: Now test maxLevel (pyramid levels) with a LARGE motion ball (shift=45px).
# Use winSize=(15,15) and vary maxLevel in [0, 1, 2, 3].
# At maxLevel=0 LK should fail. At higher levels it should recover.
# Plot: x=maxLevel, y=mean EPE at ball center.

synth_large1 = make_ball(180, 140)
synth_large2 = make_ball(180 + 45, 140)

levels = [0, 1, 2, 3]
epes = []

for lvl in levels:
    # YOUR CODE HERE
    pass

plt.figure(figsize=(6, 4))
plt.plot(levels, epes, 'o-', color='steelblue')
plt.xlabel('maxLevel (pyramid depth)')
plt.ylabel('Mean EPE at tracked corners (px)')
plt.title('LK — Effect of pyramid depth on 45px motion')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 3 — FlowNet (Pretrained)

FlowNet takes two RGB frames stacked as input and predicts a **dense** flow field $(H, W, 2)$ for every pixel.

```
Frame1 (RGB) ─┐
               ├─ (B,T=2,C,H,W) → CNN Encoder → Decoder → (H,W,2) flow
Frame2 (RGB) ─┘
```

### 3.1 — Load pretrained model

In [ ]:
import ptlflow, torch

# ptlflow v0.4+: correct argument is ckpt_path= (not pretrained_ckpt=)
flownet = ptlflow.get_model('flownets', ckpt_path='things')
flownet = flownet.to(DEVICE).eval()

n_params = sum(p.numel() for p in flownet.parameters())
print(f"FlowNetS loaded. Parameters: {n_params/1e6:.1f}M")

# Sanity check — near-zero flow on blank input confirms pretrained weights loaded
dummy = {'images': torch.zeros(1, 2, 3, 64, 64).to(DEVICE)}
with torch.no_grad():
    out = flownet(dummy)
mag = out['flows'][0, 0].abs().mean().item()
print(f"Sanity check mean |flow| on zeros: {mag:.4f}  (should be < 1.0)")

### 3.2 — Inference helper

In [ ]:
# TODO: Complete the run_flownet function.
# The model expects input dict: {'images': tensor of shape (B=1, T=2, C=3, H, W)}
# Values float32 in [0,1], RGB channel order.
# Output: preds['flows'] has shape (B, T-1, 2, H, W)
# Return the flow as a (H, W, 2) numpy array.

def run_flownet(model, bgr1: np.ndarray, bgr2: np.ndarray) -> np.ndarray:
    """Run FlowNetS on two BGR images. Returns (H, W, 2) flow."""

    def to_tensor(bgr):
        # TODO: BGR → RGB, normalize to [0,1], return (C,H,W) float32 tensor
        # YOUR CODE HERE
        pass

    t1 = to_tensor(bgr1)   # (3, H, W)
    t2 = to_tensor(bgr2)   # (3, H, W)

    # TODO: stack t1 and t2 into shape (1, 2, 3, H, W) and move to DEVICE
    images = # YOUR CODE HERE

    with torch.no_grad():
        # TODO: pass {'images': images} through model
        preds = # YOUR CODE HERE

    # TODO: extract flow from preds['flows'], shape (B,T-1,2,H,W)
    # Take first batch, first pair → (2,H,W) → permute to (H,W,2) → numpy
    flow_tensor = # YOUR CODE HERE
    return # YOUR CODE HERE


# Quick test
synth1_bgr = cv2.cvtColor(synth1, cv2.COLOR_GRAY2BGR)
synth2_bgr = cv2.cvtColor(synth2, cv2.COLOR_GRAY2BGR)
test_flow = run_flownet(flownet, synth1_bgr, synth2_bgr)
print(f"Output shape: {test_flow.shape}  (expected: {synth1.shape[0]}x{synth1.shape[1]}x2)")

### 3.3 — Run on synthetic and real frames

In [ ]:
# TODO: run FlowNet on the synthetic ball pair and print the flow at the ball center
fn_flow_synth = run_flownet(flownet, synth1_bgr, synth2_bgr)

print(f"FlowNet output shape: {fn_flow_synth.shape}")
# TODO: print u and v at pixel (row=140, col=180) and compare to GT_U, GT_V
# YOUR CODE HERE

# TODO: check background flow — sample the top-left 50×50 region (should be ~0)
bg_u = # YOUR CODE HERE
bg_v = # YOUR CODE HERE
print(f"Background flow (should be ~0): u={bg_u:.3f}, v={bg_v:.3f}")

show_side_by_side(
    (synth1, 'Frame 1'),
    (synth2, 'Frame 2'),
    (flow_to_color(fn_flow_synth), 'FlowNet Dense Flow')
)

In [ ]:
# TODO: run FlowNet on the real video pair and visualize
fn_flow_real = # YOUR CODE HERE

show_side_by_side(
    (real1,                       'Real Frame 1'),
    (flow_to_color(fn_flow_real), 'FlowNet Dense Flow'),
)

## Section 4 — Visual & Quantitative Comparison

### 4.1 — Side-by-side visualization

In [ ]:
# TODO: show all three side by side on real video:
# Input Frame 1 | LK sparse result | FlowNet dense result
# YOUR CODE HERE

### 4.2 — EPE on synthetic frames

In [ ]:
# LK EPE — evaluated only at tracked corners (fair: LK makes no prediction elsewhere)
lk_tracked_epe_vals = []
for op, np_ in zip(good_old, good_new):
    pred_u = np_[0, 0] - op[0, 0]
    pred_v = np_[0, 1] - op[0, 1]
    epe = np.sqrt((pred_u - GT_U)**2 + (pred_v - GT_V)**2)
    lk_tracked_epe_vals.append(epe)
lk_epe = float(np.mean(lk_tracked_epe_vals))

# TODO: compute FlowNet EPE over the ball region (pixels where synth1 > 100)
ball_mask = synth1 > 100

def masked_epe(flow, mask, u_gt, v_gt):
    # YOUR CODE HERE
    pass

fn_epe = masked_epe(fn_flow_synth, ball_mask, GT_U, GT_V)

print(f"Ground truth: u={GT_U}, v={GT_V}")
print(f"{'Method':<15} {'EPE':>20} {'Coverage':>15}")
print("-" * 52)
print(f"{'Lucas-Kanade':<15} {lk_epe:>20.3f} px  {'(tracked corners only)':>15}")
print(f"{'FlowNet':<15} {fn_epe:>20.3f} px  {'(every ball pixel)':>15}")